# 🚀 Data Augmentation para Facturas - Google Drive

Este notebook expande tu dataset de facturas generando múltiples variaciones mediante desplazamientos.

**¿Qué hace?**
- Lee tus facturas desde `Datos extraidos de Originales/` en Google Drive
- Genera 16 variaciones por cada factura (desplazamientos en diferentes direcciones)
- Guarda en `facturas_con_margenes_modificados/`
- **Resultado:** 1 factura → 17 archivos (1 original + 16 variaciones)

**Estructura de entrada esperada:**
```
Datos extraidos de Originales/
├── facturas_procesadas/
│   ├── IMG-20251026-WA0038.pdf
│   └── ...
└── anotaciones/
    ├── IMG-20251026-WA0038.json
    └── ...
```

**Estructura de salida:**
```
facturas_con_margenes_modificados/
├── facturas_procesadas/
│   ├── IMG-20251026-WA0038.pdf                    (original)
│   ├── IMG-20251026-WA0038_derecha_small.pdf      (variación)
│   └── ... (16 variaciones más)
└── anotaciones/
    ├── IMG-20251026-WA0038.json                   (original)
    ├── IMG-20251026-WA0038_derecha_small.json     (variación)
    └── ... (16 variaciones más)
```

## 📦 Paso 1: Instalación y Setup

In [ ]:
# Instalar dependencias
!pip install -q Pillow pdf2image numpy
!apt-get install -qq poppler-utils

print("✅ Dependencias instaladas")

In [ ]:
# Montar Google Drive
from google.colab import drive
drive.mount('/content/drive')

print("✅ Google Drive montado")

In [ ]:
# Clonar el repositorio
!git clone https://github.com/GynoRomeroPrado/modificador-de-espacios-de-facturas.git
%cd modificador-de-espacios-de-facturas

print("✅ Repositorio clonado")

In [ ]:
# ========================================
# ⚙️ CONFIGURA ESTAS RUTAS
# ========================================

# Directorio con tus facturas originales en Google Drive
# Debe contener las subcarpetas: facturas_procesadas/ y anotaciones/
INPUT_DIR = "/content/drive/MyDrive/Datos extraidos de Originales"

# Directorio donde guardar el dataset augmentado
OUTPUT_DIR = "/content/drive/MyDrive/facturas_con_margenes_modificados"

# DPI para convertir PDFs a imágenes (mayor = mejor calidad, más pesado)
DPI = 200

# Número máximo de facturas a procesar (None = todas, 1 = solo una para pruebas)
MAX_INVOICES = 1  # Cambia a None para procesar todas

print("✅ CONFIGURACIÓN:")
print(f"📁 Input:         {INPUT_DIR}")
print(f"📁 Output:        {OUTPUT_DIR}")
print(f"🔧 DPI:           {DPI}")
print(f"📊 Max facturas:  {MAX_INVOICES or 'Todas'}")

In [ ]:
# ========================================
# CONFIGURA ESTAS RUTAS
# ========================================

# Directorio con tus facturas originales (imagen + JSON)
INPUT_DIR = "/content/drive/MyDrive/Facturas"

# Directorio donde guardar el dataset augmentado
OUTPUT_DIR = "/content/drive/MyDrive/Facturas_Procesadas"

# DPI para convertir PDFs a imágenes (mayor = mejor calidad, más pesado)
DPI = 200

print(f"📁 Input:  {INPUT_DIR}")
print(f"📁 Output: {OUTPUT_DIR}")
print(f"🔧 DPI:    {DPI}")

## 🔍 Paso 3: Verificar Estructura de Entrada

In [ ]:
import os
from pathlib import Path

# Verificar que el directorio principal existe
if not os.path.exists(INPUT_DIR):
    print(f"❌ ERROR: El directorio {INPUT_DIR} no existe")
    print("\nPor favor:")
    print("1. Verifica que la ruta sea correcta")
    print("2. Asegúrate de haber montado Google Drive")
else:
    print(f"✅ Directorio encontrado: {INPUT_DIR}\n")
    
    # Verificar subdirectorios
    facturas_dir = Path(INPUT_DIR) / 'facturas_procesadas'
    anotaciones_dir = Path(INPUT_DIR) / 'anotaciones'
    
    if not facturas_dir.exists():
        print(f"❌ ERROR: No existe {facturas_dir}")
        print("   Debe existir la carpeta 'facturas_procesadas' con los PDFs")
    elif not anotaciones_dir.exists():
        print(f"❌ ERROR: No existe {anotaciones_dir}")
        print("   Debe existir la carpeta 'anotaciones' con los JSONs")
    else:
        # Contar archivos
        pdf_files = list(facturas_dir.glob('*.pdf'))
        json_files = list(anotaciones_dir.glob('*.json'))
        
        print(f"📊 Contenido:")
        print(f"  • PDFs en facturas_procesadas: {len(pdf_files)}")
        print(f"  • JSONs en anotaciones:        {len(json_files)}")
        
        # Verificar pares
        pairs = 0
        for pdf_file in pdf_files:
            json_file = anotaciones_dir / f"{pdf_file.stem}.json"
            if json_file.exists():
                pairs += 1
        
        print(f"  • Pares completos (PDF+JSON):  {pairs}")
        
        if pairs == 0:
            print("\n⚠️  ADVERTENCIA: No se encontraron pares completos")
            print("   Cada PDF debe tener su JSON correspondiente")
        else:
            facturas_a_procesar = min(pairs, MAX_INVOICES) if MAX_INVOICES else pairs
            print(f"\n✅ Listo para procesar {facturas_a_procesar} factura(s)")
            print(f"   Resultado esperado: {facturas_a_procesar} × 17 = {facturas_a_procesar * 17} archivos (PDFs + JSONs)")

## 🚀 Paso 4: Procesar Dataset

Este paso genera todas las variaciones augmentadas.

In [ ]:
import sys
sys.path.append('/content/modificador-de-espacios-de-facturas/src')

from main import InvoiceDatasetAugmenter

# Crear augmenter
augmenter = InvoiceDatasetAugmenter(
    input_dir=INPUT_DIR,
    output_dir=OUTPUT_DIR,
    dpi=DPI,
    max_invoices=MAX_INVOICES
)

# Procesar dataset
stats = augmenter.process_dataset()

## ✅ Paso 5: Verificar Resultados

In [ ]:
import json

# Leer reporte
report_path = os.path.join(OUTPUT_DIR, 'dataset_report.json')

if os.path.exists(report_path):
    with open(report_path, 'r') as f:
        report = json.load(f)
    
    print("📊 REPORTE DEL PROCESO")
    print("=" * 60)
    print(f"\n📅 Fecha: {report['timestamp']}")
    print(f"\n📁 Directorios:")
    print(f"  • Input:  {report['input_directory']}")
    print(f"  • Output: {report['output_directory']}")
    print(f"\n📊 Estadísticas:")
    print(f"  • Facturas originales:  {report['statistics']['original_invoices']}")
    print(f"  • Facturas augmentadas: {report['statistics']['augmented_invoices']}")
    print(f"  • Total de facturas:    {report['statistics']['total_invoices']}")
    
    if report['statistics']['errors']:
        print(f"\n⚠️  Errores: {len(report['statistics']['errors'])}")
        for error in report['statistics']['errors']:
            print(f"    - {error}")
    else:
        print("\n✅ Sin errores")
    
    print("\n" + "=" * 60)
else:
    print("❌ No se encontró el reporte")

## 👀 Paso 6: Explorar Resultados (Opcional)

In [ ]:
# Listar archivos generados
facturas_dir = os.path.join(OUTPUT_DIR, 'facturas_procesadas')
anotaciones_dir = os.path.join(OUTPUT_DIR, 'anotaciones')

print("📁 ARCHIVOS GENERADOS\n")

if os.path.exists(facturas_dir):
    pdf_files = list(Path(facturas_dir).glob('*.pdf'))
    print(f"📂 facturas_procesadas/ ({len(pdf_files)} PDFs)")
    for f in sorted(pdf_files)[:20]:  # Mostrar primeros 20
        print(f"  • {f.name}")
    if len(pdf_files) > 20:
        print(f"  ... y {len(pdf_files) - 20} más")

print()

if os.path.exists(anotaciones_dir):
    json_files = list(Path(anotaciones_dir).glob('*.json'))
    print(f"📂 anotaciones/ ({len(json_files)} JSONs)")
    for f in sorted(json_files)[:20]:  # Mostrar primeros 20
        print(f"  • {f.name}")
    if len(json_files) > 20:
        print(f"  ... y {len(json_files) - 20} más")

## 🖼️ Paso 7: Visualizar Ejemplo (Opcional)

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
from pdf2image import convert_from_path

# Buscar PDFs augmentados
facturas_dir = os.path.join(OUTPUT_DIR, 'facturas_procesadas')
augmented_pdfs = [f for f in sorted(Path(facturas_dir).glob('*.pdf')) 
                  if '_' in f.stem][:4]  # PDFs con _ son augmentados

if augmented_pdfs:
    fig, axes = plt.subplots(2, 2, figsize=(15, 15))
    axes = axes.flatten()
    
    for idx, pdf_path in enumerate(augmented_pdfs):
        # Convertir PDF a imagen
        images = convert_from_path(str(pdf_path), dpi=100, first_page=1, last_page=1)
        img = images[0]
        
        axes[idx].imshow(img)
        axes[idx].set_title(pdf_path.name, fontsize=10)
        axes[idx].axis('off')
    
    plt.tight_layout()
    plt.show()
    print("\n✅ Ejemplos de facturas augmentadas")
else:
    print("❌ No se encontraron PDFs augmentados")
    print("Tip: Los archivos augmentados tienen nombres como 'IMG-20251026-WA0038_derecha_small.pdf'")

## 🎯 Próximos Pasos

1. ✅ Tu dataset ha sido expandido exitosamente
2. 📁 Los archivos están en tu Google Drive en: `{OUTPUT_DIR}`
3. 🚀 Ahora puedes usar este dataset expandido para:
   - Entrenar tu modelo de detección
   - Mejorar la precisión del sistema
   - Generar más datos de entrenamiento

**Resultado:**
```
facturas_con_margenes_modificados/
├── facturas_procesadas/         # PDFs (originales + variaciones)
│   ├── IMG-20251026-WA0038.pdf                    (original)
│   ├── IMG-20251026-WA0038_derecha_small.pdf      (variación)
│   ├── IMG-20251026-WA0038_izquierda_small.pdf    (variación)
│   ├── IMG-20251026-WA0038_abajo_small.pdf        (variación)
│   └── ... (13 variaciones más)
│
├── anotaciones/                 # JSONs (originales + copias)
│   ├── IMG-20251026-WA0038.json                   (original)
│   ├── IMG-20251026-WA0038_derecha_small.json     (copia)
│   └── ... (16 copias más)
│
└── dataset_report.json          # Reporte del proceso
```

**16 Transformaciones aplicadas:**
- Horizontal: derecha/izquierda (small, medium, large)
- Vertical: arriba/abajo (small, medium, large)
- Diagonales: 4 direcciones (small)

**Total por factura:** 1 original + 16 variaciones = 17 archivos